# River data exploration

First look at the ARSO hourly water level and discharge data in `data/reke/`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
predhodnice_aktivnih_postaj = pd.read_csv('data/reke/aktivneVP_predhodnice.csv')
predhodnice_aktivnih_postaj = dict(map(lambda x: (x[0],list(x[1].values())),predhodnice_aktivnih_postaj.set_index('VPSIF').to_dict(orient='index').items()))
print("Stevilo aktivnih postaj: ",len(predhodnice_aktivnih_postaj))

In [ ]:
st_predhodnic ={}
for v in predhodnice_aktivnih_postaj.values():
    st = sum([1 if val != -1 else 0 for val in v])
    st_predhodnic[st] = st_predhodnic.get(st,0) + 1
st_predhodnic

#plt.bar(st_predhodnic.keys(),st_predhodnic.values())
#plt.xticks(list(st_predhodnic.keys()))
#plt.yticks(np.arange(0, max(st_predhodnic.values())+20, 20))
#plt.show()
#   

In [ ]:
predhodnice_aktivnih_postaj.keys()

In [ ]:
import json
hid_arhiv = json.load(open('data/reke/POV/POV/hid_arhiv.geojson','r'))
#[v  for v in hid_arhiv['features'] if int(v['properties']['id']) == 1060][0]
# create a dictionary where key is the id of the station and value is the name of the station
hid_postaje = {int(v['properties']['id']):v['properties']['name'] for v in hid_arhiv['features']}
#hid_postaje

In [ ]:
#[v  for v in hid_arhiv['features'] if int(v['properties']['id']) == 1060][0]

aktivne_postaje = predhodnice_aktivnih_postaj.keys()

In [ ]:
# preberimo podatke o aktivnih postajah. Zdruzimo podatke o aktivnih postajah s podatki njihovih predhodnic.
import os
import re

postaja2files = {}

data_dir = 'data/reke/POV/POV/'

vse_postaje = list(filter(lambda x: re.match(r'[0-9]+',x), os.listdir(data_dir)))
print("Vse postaje: ",(vse_postaje))

# here we collect files containing hourly data for each of the active stations 
for postaja in vse_postaje:
    postaja_files = []
    #print(data_dir+str(postaja)+'/Q/urne/')
    for dirpath, dirnames, filenames in os.walk(data_dir+str(postaja)+'/H/urne/'):
       # print(dirpath, dirnames, filenames)
        # check if filename staisfies regular expression => we want that file contains hourly data
        for filename in filenames:
            if re.match(rf'{postaja}_H_urne_[1-2][0,9][0-9][0-9].csv', filename):
                postaja_files.append(dirpath +"/"+ filename)
    #print(postaja_files)
    postaja2files[int(postaja)] = postaja_files

print("?????",postaja2files.keys())

# concat files of old stations to the active stations
for postaja, predhodnice in predhodnice_aktivnih_postaj.items():
    for predhodnica in predhodnice:
        if predhodnica != -1:
            if predhodnica in postaja2files:
                postaja2files[postaja] += postaja2files[predhodnica]
            else:
                print("Nimamo podatkov za postajo: ",predhodnica," ki je predhodnica postaje: ",postaja)
                
# remove all entries that are not active stations (copy only active stations to new dictionary)
new_postaja2files = {}
for postaja in postaja2files.keys():
    if postaja in aktivne_postaje:
        new_postaja2files[postaja] = postaja2files[postaja]
postaja2files = new_postaja2files

# check if we have all the files for each active station
del_list = []
for postaja,files in postaja2files.items():
    if len(files) == 0:
        print("Postaja: ",postaja," nima urnih podatkov")
        del_list.append(postaja)
for x in del_list:
    del postaja2files[x]

# how many years of data we have for each station, plot a histogram
st_let = {}
for postaja,files in postaja2files.items():
    st_let[postaja] = len(files)

plt.figure(figsize=(20, 10))
plt.hist(st_let.values(),bins=range(0,max(st_let.values())+1,1))
plt.xticks(range(0,max(st_let.values())+1,1))
plt.title('Stevilo let podatkov za aktivne postaje (aktivnim pripnemo tudi podatke starih postaj)')
plt.xlabel('Stevilo let')
plt.ylabel('Stevilo postaj')
plt.show()

st_let.items()


In [ ]:
# read data for each active station
postaja2data = {}
for postaja,files in postaja2files.items():
    if postaja != 4230:
        continue
    data = []
    for file in files:
        data.append(pd.read_csv(file,sep=';'))
    postaja2data[postaja] = pd.concat(data)

In [ ]:
for postaja,data in postaja2data.items():
    # set index to datetime
    data['datum'] = pd.to_datetime(data['datum'])
    postaja2data[postaja] = data.set_index('datum').sort_index().rename(columns={'vrednost':f'{postaja}_Q'})
    # remove duplicates, based on index value
    postaja2data[postaja] = postaja2data[postaja][~postaja2data[postaja].index.duplicated(keep='first')]

In [ ]:
# draw a plot showing for each hour in a day how many stations have data
#plt.figure(figsize=(20, 10))
#postaja2data[1060].columns

In [ ]:
# how many nan values we have for each station
st_nan = {}
for postaja,data in postaja2data.items():
    st_nan[postaja] = data.isna().sum().sum()
print("max nan: ",max(st_nan.values()))

In [ ]:
#postaja2data[4200].loc[:'2020-01-01 00:00:00'].to_csv('temp_data_4200.csv')

In [ ]:
for leto in range(1990,2020):
    print(leto)
    print(sum([(postaja2data[k].loc[f'{leto}-01-01 00:00:00':f'{leto}-12-31 23:00:00'].shape[0]/(365*24)) >= 0.7 for k in postaja2data.keys()]))
    

for leto in [2014]:
    print(([k for k in postaja2data.keys() if (postaja2data[k].loc[f'{leto}-01-01 00:00:00':f'{leto}-12-31 23:00:00'].shape[0]/(365*24)) >= 0.5]))
print(len(postaja2data.keys()))
print(len(([k for k in postaja2data.keys() if (postaja2data[k].loc[f'{leto}-01-01 00:00:00':f'{leto}-12-31 23:00:00'].shape[0]/(365*24)) >= 0.8])))

In [ ]:
kandidatke = [1140, 2880, 3850, 5078, 6020, 6060, 6140, 6200, 6240, 6415, 6550, 6630, 6720, 7160, 8080, 8450, 8500]

In [ ]:
for p in kandidatke:
    print(">>>",p)
    for leto in range(1990,2020):

        if not postaja2data[p].loc[f'{leto}-01-01 00:00:00':f'{leto}-12-31 23:00:00'].shape[0]/(365*24) >= 0.7:
            print(leto)

In [ ]:
izbrane_postaje = [8500,8450,8080,7160,6720,6630,6550,6415,6240,6200,6060,6020,5078,3850,2880]

In [ ]:
#
#
#
# prepare dateset for training -> napovedi za x ur naprej
#
#
#
#
needed_hours = [6,18]
napoved_za_x_ur = 12
izbrane_postaje = [8500,8450,8080,7160,6720]

tmps = []

for postaja, data in postaja2data.items():
    if postaja not in izbrane_postaje:
        continue
    tmp = data.loc['1990':].copy()
    # change tiem from utc+1 to utc
    tmp.index = tmp.index - pd.DateOffset(hours=1)
    # add columns for each hour
    for h in range(1,napoved_za_x_ur+1):
        tmp[f'{postaja}_Q_{h}'] = tmp[f'{postaja}_Q'].shift(-h)
    # drop original column
    tmp = tmp.drop(columns=[f'{postaja}_Q'])
    # select only rows where index is in needed_hours
    tmp = tmp[tmp.index.hour.isin(needed_hours)]

    tmps.append(tmp)

postaje_dataset = pd.concat(tmps,axis=1,sort=True,join='outer')
#postaje_dataset.to_csv('data/reke/postaje_dataset_test.csv')
#postaje_dataset


In [ ]:
#dd = pd.read_csv('data/reke/postaje_dataset_test.csv',index_col=0)

In [ ]:
#
#
#
# prepare dateset for training -> give past 48 hours of data
#
#
#
#
needed_hours = [0,6,12,18]
x_ur_za_nazaj = 120
izbrane_postaje = [4200, 4230, 4298]

tmps = []

for postaja, data in postaja2data.items():
    if postaja not in izbrane_postaje:
        continue
    tmp = data.loc['1990':].copy()
    # change tiem from utc+1 to utc
    tmp.index = tmp.index - pd.DateOffset(hours=1)
    # add columns for each hour
    for h in range(0,x_ur_za_nazaj):
        tmp[f'{postaja}_Q_{h}'] = tmp[f'{postaja}_Q'].shift(h)
    # drop original column
    tmp = tmp.drop(columns=[f'{postaja}_Q'])
    # select only rows where index is in needed_hours
    tmp = tmp[tmp.index.hour.isin(needed_hours)]

    tmps.append(tmp)

postaje_dataset = pd.concat(tmps,axis=1,sort=True,join='outer')
print(len(list(postaje_dataset.index)))
#len(list(postaje_dataset.dropna().index))
#postaje_dataset.to_csv('data/reke/postaje_dataset_test_history.csv')

In [ ]:
import copy
#postaja2dataQ = copy.deepcopy(postaja2data)
#postaja2dataH = copy.deepcopy(postaja2data)

In [ ]:
### PLOTS ####

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 18})

postaja = 4230
fig, ax1 = plt.subplots(figsize=(20, 10))

# Pretok (Q)
data_Q = postaja2dataQ[postaja].loc['2000':'2025'][f'{postaja}_Q'].rolling(365*24, center=False, min_periods=365*24).mean()
lns1 = ax1.plot(data_Q, color='tab:blue', label='Pretok [m$^3$/s]')
ax1.set_xlabel('Datum')
ax1.set_ylabel('Pretok [m$^3$/s]', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

# Višina vodostaja (H) - predpostavljamo, da je stolpec {postaja}_H
ax2 = ax1.twinx()
data_H = postaja2dataH[postaja].loc['2000':'2025'][f'{postaja}_Q'].rolling(365*24, center=False, min_periods=365*24).mean()
lns2 = ax2.plot(data_H, color='tab:orange', label='Višina vodostaja [cm]')
ax2.set_ylabel('Višina vodostaja [cm]', color='tab:orange')
ax2.tick_params(axis='y', labelcolor='tab:orange')

# Title and legend
plt.title(f"Pretok in višina vodostaja pri postaji Zminec na Poljanski Sori - enoletno glajenje")
lns = lns1 + lns2
labels = [l.get_label() for l in lns]
ax1.legend(lns, labels, loc='upper left')

plt.show()


In [ ]:

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 18})
# plot time series data for station 4230
postaja = 4230
plt.figure(figsize=(20, 10))
# Popravi za višino vodostaja (H), predpostavljamo, da je stolpec {postaja}_H
data = postaja2dataH[postaja].loc['2000':'2025'][f'{postaja}_Q'].rolling(365*24, center=False, min_periods=365*24).mean()
plt.plot(data)
#plt.title(f'Postaja: {postaja}')

plt.title(f"Višina vodostaja pri postaji Zminec na Poljanski Sori - enoletno glajenje")
plt.xlabel('Datum')
plt.ylabel('Višina vodostaja [cm]')